In [ ]:
# ONE-CELL ENSEMBLE TRAINING: ViT + ConvNeXt + EfficientNetV2 on Plant Leaf Disease dataset

import os, zipfile, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ------------- CONFIG -------------
ZIP_URL   = "https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/b4e3a32f-c0bd-4060-81e9-6144231f2520/file_downloaded"
ZIP_PATH  = "datasets.zip"
EXTRACT_DIR = "datasets"
BATCH_SIZE = 32        # reduce to 16 if you get CUDA OOM
IMG_SIZE   = 224
EPOCHS     = 2         # start small; increase if you have time
VAL_TEST_FRACTION = 0.2  # 80/10/10 split
RANDOM_SEED = 42
NUM_WORKERS = 2

MODEL_NAMES = {
    "vit"      : "vit_base_patch16_224",
    "convnext" : "convnext_tiny",
    "effnetv2" : "efficientnetv2_rw_s",
}

# ------------- SETUP -------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Install timm if needed
try:
    import timm
except ImportError:
    # !pip install -q timm
    import timm

# ------------- DOWNLOAD & UNZIP -------------
if not os.path.exists(ZIP_PATH):
    print("Downloading dataset (~900MB)...")
    !wget -O "datasets.zip" "$ZIP_URL"
else:
    print("ZIP already exists, skipping download.")

if not os.path.exists(EXTRACT_DIR):
    print("Unzipping...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Unzipped to:", EXTRACT_DIR)
else:
    print("Already unzipped, skipping.")

DATA_ROOT = Path(EXTRACT_DIR) / "Plant_leave_diseases_dataset_with_augmentation"
assert DATA_ROOT.exists(), f"Data root not found: {DATA_ROOT}"
print("Data root:", DATA_ROOT)

# ------------- INDEX DATA -------------
img_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".JPG")

filepaths, labels = [], []
for cls_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    for ext in img_exts:
        for img_path in cls_dir.glob(f"*{ext}"):
            filepaths.append(str(img_path))
            labels.append(cls_dir.name)

df = pd.DataFrame({"filepath": filepaths, "label": labels})
print("Total images:", len(df))
print("Classes:", df["label"].nunique())

class_names = sorted(df["label"].unique())
label2idx = {c: i for i, c in enumerate(class_names)}
idx2label = {i: c for c, i in label2idx.items()}
df["label_idx"] = df["label"].map(label2idx)

# Optional: quick class counts
print("Top 10 classes by count:")
print(df["label"].value_counts().head(10))

# ------------- TRAIN/VAL/TEST SPLIT -------------
train_df, temp_df = train_test_split(
    df,
    test_size=VAL_TEST_FRACTION,
    stratify=df["label_idx"],
    random_state=RANDOM_SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_idx"],
    random_state=RANDOM_SEED,
)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# ------------- DATASET & DATALOADERS -------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class PlantDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.paths = dataframe["filepath"].values
        self.labels = dataframe["label_idx"].values.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = PlantDataset(train_df, transform=train_transform)
val_dataset   = PlantDataset(val_df,   transform=eval_transform)
test_dataset  = PlantDataset(test_df,  transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(device.type=="cuda"))
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=(device.type=="cuda"))
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=(device.type=="cuda"))

num_classes = len(class_names)
print("num_classes:", num_classes)

# ------------- TRAINING FUNCTION -------------  
def train_one_model(model_name, train_loader, val_loader, epochs=EPOCHS):
    print(f"\n===== Training {model_name} =====")
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))

    best_acc = 0.0
    best_state = None

    for epoch in range(1, epochs+1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * labels.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Epoch {epoch}/{epochs} | "
              f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = model.state_dict()

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"Best val acc for {model_name}: {best_acc:.4f}")
    return model, best_acc

# ------------- TRAIN ALL MODELS -------------
trained_models = []
val_scores = {}

for nick, mname in MODEL_NAMES.items():
    model, best_acc = train_one_model(mname, train_loader, val_loader, epochs=EPOCHS)
    trained_models.append(model)
    val_scores[mname] = best_acc

print("\nValidation accuracies:")
for mname, acc in val_scores.items():
    print(f"{mname}: {acc:.4f}")

# ------------- EVALUATION HELPERS -------------
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_targets.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

    all_targets = np.concatenate(all_targets)
    all_preds = np.concatenate(all_preds)
    acc = correct / total
    return acc, all_targets, all_preds

def evaluate_ensemble(models, loader):
    for m in models:
        m.eval()
    correct = 0
    total = 0
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            probs_sum = None
            for m in models:
                outputs = m(imgs)
                probs = torch.softmax(outputs, dim=1)
                probs_sum = probs if probs_sum is None else probs_sum + probs
            avg_probs = probs_sum / len(models)
            preds = avg_probs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_targets.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

    all_targets = np.concatenate(all_targets)
    all_preds = np.concatenate(all_preds)
    acc = correct / total
    return acc, all_targets, all_preds

# ------------- SINGLE-MODEL TEST ACCURACY -------------
print("\nSingle-model test accuracies:")
for nick, model in zip(MODEL_NAMES.keys(), trained_models):
    acc, _, _ = evaluate_model(model, test_loader)
    print(f"{nick}: {acc:.4f}")

# ------------- ENSEMBLE TEST ACCURACY -------------
ens_acc, y_true, y_pred = evaluate_ensemble(trained_models, test_loader)
print(f"\nEnsemble test accuracy: {ens_acc:.4f}")

# Optional: classification report for ensemble
print("\nEnsemble classification report:")
print(classification_report(
    y_true, y_pred,
    target_names=[idx2label[i] for i in range(num_classes)],
    digits=3
))

Device: cuda


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

--2025-12-07 16:01:09--  https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/b4e3a32f-c0bd-4060-81e9-6144231f2520/file_downloaded
Resolving data.mendeley.com (data.mendeley.com)... 162.159.130.86, 162.159.133.86
Connecting to data.mendeley.com (data.mendeley.com)|162.159.130.86|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/349ac012-2948-4172-bbba-3bf8f76596fd [following]
--2025-12-07 16:01:10--  https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/349ac012-2948-4172-bbba-3bf8f76596fd
Resolving prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com (prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com)... 52.92.3.2, 3.5.67.210, 52.218.46.58, ...
Connecting to prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com (prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com)|52.92.3.2|:443... conn

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/tmp/ipykernel_20/1477604687.py:177: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
/tmp/ipykernel_20/1477604687.py:192: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
/tmp/ipykernel_20/1477604687.py:217: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):


Epoch 1/2 | train_loss=0.1817 acc=0.9454 | val_loss=0.0654 acc=0.9777
Epoch 2/2 | train_loss=0.0778 acc=0.9753 | val_loss=0.0835 acc=0.9719
Best val acc for vit_base_patch16_224: 0.9777

===== Training convnext_tiny =====


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1/2 | train_loss=0.1709 acc=0.9503 | val_loss=0.0739 acc=0.9761
Epoch 2/2 | train_loss=0.0529 acc=0.9842 | val_loss=0.0784 acc=0.9759
Best val acc for convnext_tiny: 0.9761

===== Training efficientnetv2_rw_s =====


model.safetensors:   0%|          | 0.00/96.5M [00:00<?, ?B/s]

Epoch 1/2 | train_loss=0.2957 acc=0.9218 | val_loss=0.1942 acc=0.9423
Epoch 2/2 | train_loss=0.0309 acc=0.9901 | val_loss=0.1232 acc=0.9667
Best val acc for efficientnetv2_rw_s: 0.9667

Validation accuracies:
vit_base_patch16_224: 0.9777
convnext_tiny: 0.9761
efficientnetv2_rw_s: 0.9667

Single-model test accuracies:
vit: 0.9680
convnext: 0.9735
effnetv2: 0.9616

Ensemble test accuracy: 0.9917

Ensemble classification report:
                                               precision    recall  f1-score   support

                           Apple___Apple_scab      1.000     1.000     1.000       100
                            Apple___Black_rot      1.000     1.000     1.000       100
                     Apple___Cedar_apple_rust      1.000     1.000     1.000       100
                              Apple___healthy      1.000     0.994     0.997       164
                    Background_without_leaves      1.000     1.000     1.000       114
                          Blueberry___healthy  

In [2]:
import os
import torch

SAVE_DIR = "plant_disease_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# val_scores keys are: "vit_base_patch16_224", "convnext_tiny", "efficientnetv2_rw_s"
print("Saving models to:", SAVE_DIR)
for model_name, model in zip(val_scores.keys(), trained_models):
    path = os.path.join(SAVE_DIR, f"{model_name}_best.pth")
    torch.save({
        "model_name": model_name,
        "state_dict": model.state_dict(),
        "class_names": class_names,
    }, path)
    print("  ->", path)

Saving models to: plant_disease_models
  -> plant_disease_models/vit_base_patch16_224_best.pth
  -> plant_disease_models/convnext_tiny_best.pth
  -> plant_disease_models/efficientnetv2_rw_s_best.pth
